# Eurosat Assignment

**Authors:** Jason Fan, Henry Sywulak-Herr

## 1. Data Loading, Processing, and Exploration

### 1.1 Data Preparation

Visit the EuroSAT data description page and download the data. Perform basic exploratory data analysis, assessing the class distribution across the dataset and plotting one image from each class in a 2x5 grid.

Flatten the images into a 2D data matrix (n x p, where n is the number of samples and p is the number of pixels in each image). Load these and the labels into numpy arrays. Split the data into training (60%) and testing (40%) datasets, stratified on class labels (so that there is an equal percentage of each class type in each of the training and testing sets).

Lastly, create a grayscale version of this dataset. You will use this for the traditional machine learning models and the first couple of deep learning models.

In [1]:
import os
import io
import zipfile

import requests

import rioxarray as rxr
import xarray as xr
import rasterio
import pandas as pd

In [ ]:
BASE_PATH = "./eurosat_files/"
ZIP_PATH = "./eurosat_files/eurosat.zip"

if os.path.exists(BASE_PATH):
    print("Eurosat data already downloaded")
else:
    os.makedirs(BASE_PATH, exist_ok=True)

    URL_EUROSAT = "https://zenodo.org/records/7711810/files/EuroSAT_MS.zip?download=1"

    print(f"Downloading from: {URL_EUROSAT}...")

    with requests.get(URL_EUROSAT, stream=True, timeout=30) as r:
        r.raise_for_status()
        with open(ZIP_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)

    print("Download complete")

    print("Extracting files...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(BASE_PATH)

    print("Extraction complete")

Eurosat data already downloaded


: 

Flatten the images into a 2D data matrix (n x p, where n is the number of samples and p is the number of pixels in each image). Load these and the labels into numpy arrays. Split the data into training (60%) and testing (40%) datasets, stratified on class labels (so that there is an equal percentage of each class type in each of the training and testing sets).

Lastly, create a grayscale version of this dataset. You will use this for the traditional machine learning models and the first couple of deep learning models.

In [ ]:
import os
import glob
import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt
import random

BASE_PATH = "./eurosat_files/"
dataset_dir = os.path.join(BASE_PATH, "EuroSAT_MS")
classes = sorted([d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))])
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(classes)}
idx_to_class = {idx: cls_name for cls_name, idx in class_to_idx.items()}

# ==========================================
# 1. MEMORY-SAFE DATA LOADING
# ==========================================
print("Counting images to pre-allocate memory...")
image_paths = []
labels = []

for cls_name in classes:
    cls_dir = os.path.join(dataset_dir, cls_name)
    paths = glob.glob(os.path.join(cls_dir, "*.tif"))
    for p in paths:
        image_paths.append(p)
        labels.append(class_to_idx[cls_name])

total_images = len(image_paths)
print(f"Total original images: {total_images}")

# Determine how many images we will augment (20%)
num_to_augment = int(total_images * 0.20)
total_final_images = total_images + num_to_augment

print(f"Allocating memory for {total_final_images} images... (This prevents crashes)")
# Pre-allocate a massive array of zeros. 
# Using float32 cuts memory usage in half compared to Python's default float64
X_full = np.zeros((total_final_images, 64, 64, 13), dtype=np.float32)
y_full = np.zeros(total_final_images, dtype=np.int32)

print("Loading original images into memory...")
for i, path in enumerate(image_paths):
    X_full[i] = tiff.imread(path)
    y_full[i] = labels[i]

# ==========================================
# 2. APPLY DATA AUGMENTATION IN-PLACE
# ==========================================
print(f"Applying horizontal flip to {num_to_augment} images...")

# Pick random indices from the original dataset to augment
augment_indices = random.sample(range(total_images), num_to_augment)

# Add the augmented images to the end of our pre-allocated array
current_insert_idx = total_images
for idx in augment_indices:
    # Read the original image from our array, flip it horizontally
    flipped_img = np.fliplr(X_full[idx])
    
    # Store it in the empty slots at the end of the array
    X_full[current_insert_idx] = flipped_img
    y_full[current_insert_idx] = y_full[idx]
    
    current_insert_idx += 1

print(f"Total size of the new augmented dataset: {X_full.shape[0]}")

# ==========================================
# 3. PLOT 3 RANDOM IMAGES & NEW HISTOGRAM
# ==========================================
# Plot 3 random images
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle("Three Random Images from Augmented Dataset (Band 1)", fontsize=14)

random_indices = random.sample(range(len(X_full)), 3)
for i, r_idx in enumerate(random_indices):
    img = X_full[r_idx]
    label_name = idx_to_class[y_full[r_idx]]
    axes[i].imshow(img[:, :, 0], cmap='gray') # Show just the first band
    axes[i].set_title(f"Class: {label_name}\nIndex: {r_idx}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

# Plot histogram of the newly augmented dataset
unique_labels, counts = np.unique(y_full, return_counts=True)
label_names = [idx_to_class[lbl] for lbl in unique_labels]

plt.figure(figsize=(10, 5))
plt.bar(label_names, counts, color='lightgreen', edgecolor='black')
plt.title("EuroSAT Class Distribution (After Augmentation)")
plt.xlabel("Land Cover Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


Counting images to pre-allocate memory...
Total original images: 27000
Allocating memory for 32400 images... (This prevents crashes)
Loading original images into memory...


### 1.2 Data Augmentation

Do 1.1 again, but, before splitting the data into training and testing sets or doing any preprocessing, apply data augmentation to increase the size of the dataset, appending the new samples to the original dataset. Indicate the augmentation approach(es) that you used and the total size of the new dataset. Again, plot three random images and a histogram of the label distribution across the full dataset.

In [1]:
import os
import glob
import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt
import random
from sklearn.model_selection import train_test_split

BASE_PATH = "./eurosat_files/"
dataset_dir = os.path.join(BASE_PATH, "EuroSAT_MS")
classes = sorted([d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))])
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(classes)}
idx_to_class = {idx: cls_name for cls_name, idx in class_to_idx.items()}

# 1. LOAD ORIGINAL IMAGES

image_paths = []
labels = []

for cls_name in classes:
    cls_dir = os.path.join(dataset_dir, cls_name)
    paths = glob.glob(os.path.join(cls_dir, "*.tif"))
    for p in paths:
        image_paths.append(p)
        labels.append(class_to_idx[cls_name])

total_images = len(image_paths)
print(f"Total original images: {total_images}")

X_orig = np.zeros((total_images, 64, 64, 13), dtype=np.float32)
y_orig = np.zeros(total_images, dtype=np.int32)

for i, path in enumerate(image_paths):
    X_orig[i] = tiff.imread(path)
    y_orig[i] = labels[i]

# 2. AUGMENTATION (BEFORE SPLITTING)

num_to_augment = int(total_images * 0.20)
total_final = total_images + num_to_augment

X_full = np.zeros((total_final, 64, 64, 13), dtype=np.float32)
y_full = np.zeros(total_final, dtype=np.int32)

X_full[:total_images] = X_orig
y_full[:total_images] = y_orig

augment_indices = random.sample(range(total_images), num_to_augment)
for i, idx in enumerate(augment_indices):
    X_full[total_images + i] = np.fliplr(X_orig[idx])
    y_full[total_images + i] = y_orig[idx]

print(f"Augmented dataset size: {X_full.shape[0]}")


# 3. PLOT 3 RANDOM IMAGES

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle("Three Random Images from Augmented Dataset (Band 1)", fontsize=14)

for i, r_idx in enumerate(random.sample(range(len(X_full)), 3)):
    axes[i].imshow(X_full[r_idx][:, :, 0], cmap='gray')
    axes[i].set_title(f"Class: {idx_to_class[y_full[r_idx]]}\nIndex: {r_idx}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()


# 4. HISTOGRAM OF AUGMENTED DATASET

unique_labels, counts = np.unique(y_full, return_counts=True)
label_names = [idx_to_class[lbl] for lbl in unique_labels]

plt.figure(figsize=(10, 5))
plt.bar(label_names, counts, color='lightgreen', edgecolor='black')
plt.title("EuroSAT Class Distribution (After Augmentation)")
plt.xlabel("Land Cover Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# 5. TRAIN/TEST SPLIT (AFTER AUGMENTATION)

X_flat = X_full.reshape(X_full.shape[0], -1)

X_train, X_test, y_train, y_test = train_test_split(
    X_flat, y_full, test_size=0.4, random_state=42, stratify=y_full
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size:  {X_test.shape[0]}")


# 6. GRAYSCALE VERSION

# EuroSAT band order: B02=Blue, B03=Green, B04=Red (0-indexed: 1, 2, 3)
R = X_full[:, :, :, 3]
G = X_full[:, :, :, 2]
B = X_full[:, :, :, 1]

X_gray = (0.2989 * R + 0.5870 * G + 0.1140 * B)
X_gray_flat = X_gray.reshape(X_gray.shape[0], -1)

X_train_gray, X_test_gray, y_train_gray, y_test_gray = train_test_split(
    X_gray_flat, y_full, test_size=0.4, random_state=42, stratify=y_full
)

print(f"Grayscale training set size: {X_train_gray.shape[0]}")
print(f"Grayscale testing set size:  {X_test_gray.shape[0]}")

Total original images: 27000


: 

## 2. Traditional Machine Learning

For this section, focus on three categories: "Forest (F)", "Residential (R)", and "Industrial (I)". Make sure to subset the grayscale dataset, selecting only these three classes.

### 2.1 Binary Support Vector Machine

Implement three binary SVM classifiers (use a linear kernel and default parameters) to classify [F vs R], [F vs I], and [R vs I]. Report the accuracy of each classifier, plot their ROC curves, calculate the AUCs, and show one image that is mis-classified by each classifier, including both the predicted label and the ground truth.

### 2.2 Multiclass, Majority-Vote SVM

Combine the three SVM models trained in the previous section to create a three-class classifier. The combined model will apply each one of the 3 classifiers on the testing data and will apply majority voting to decide the final class of the test sample. Again, calculate the accuracy, ROC, and AUC, and show a mis-classified image from each class, including both the predicted label and the ground truth.

### 2.3 Multiclass Random Forest

Train a Random-Forest classifier to classify the data into one of the three classes. Use the training data. Apply the trained model on testing data. Report the accuracy, plot the confusion matrix, and print a mis-classified image from each class, including both the predicted label and the ground truth.

## 3. Deep Learning

For this section, we will use the full range of possible land cover categories, so do not filter the training and testing datasets for only certain labels.

### 3.1 Greyscale Images

For this section, use the same greyscale images that you used in the traditional machine learning section.

#### 3.1.1 Single-Layer Neural Network

Implement a first deep learning model using a fully connected network with a single fully connected layer (i.e., input layer + fully connected layer as the output layer). Visualize the network architecture. (Refer to https://faroit.com/keras-docs/2.0.8/visualization/ to see the import command and function needed to visualize the architecture.) Calculate classification accuracy on the test data. (Hint: what kind of pre-processing might be necessary so that this model and the subsequent ones can handle categorical labels? Why?)

#### 3.1.2 Two-Layer Neural Network

Implement a second deep learning model adding an additional fully connected hidden layer (with an arbitrary number of nodes) to the previous model. Visualize the network architecture. Calculate classification accuracy on the test data. How did adding an additional hidden layer affect your model's performance? Why might additional hidden layers improve or potentially worsen accuracy?

#### 3.1.3 Four-Layer Neural Network with Dropout

Implement a third deep learning model adding two additional fully connected hidden layers (with arbitrary number of nodes) for a total of four, as well as drop-out layers to the previous model. Visualize the network architecture. Calculate classification accuracy on the test data. What did you observe about the impact of dropout layers on the model's performance? Explain how dropout helps in model training and under what circumstances it might be more or less effective.

#### 3.1.4 Model Comparison and Ensemble

Compare models one through three. Which network had the most parameters to learn, and by what margin? Which model was the "best"? Why? For each model, what is the impact of increasing the number of training epochs?

Implement an ensemble model that incorporates the predictions of models one through three. Calculate its classification accuracy on the test data. How does this compare to the accuracies of the three individual models? Describe the ensemble approach you implemented. Why might ensembling improve model accuracy compared to the individual models?

### 3.2 RGB Images

For this section, use the original RGB images.

#### 3.2.1 CNN Model

Implement a fourth deep learning model, a convolution neural network (CNN) that includes the following layers: Conv2D, MaxPooling2D, Dropout, Flatten, Dense. Visualize the network architecture. Calculate classification accuracy on the test data. Compare against previous models. Which model was the "best"? Why? Did you notice any limitations in terms of training speed compared to the previous models?

How does the CNN model handle spatial information differently than the fully connected models? What implications does this have for image classification? Compare the training speed of CNNs with the fully connected networks. Why do CNNs generally require more computational resources?

#### 3.2.2 Advanced Model

Implement a fifth deep learning model targeting accuracy that will outperform all previous models. You are free to use any tools and techniques, including ensemble models and pre-trained models for transfer learning. Calculate classification accuracy on the test data. What specific tools or techniques did you choose to improve accuracy? Why did you select these approaches over others? Compare against previous models. Which model was the "best"? Why?

What are the two classes with the highest labeling error? Explain using data and showing mis-classified examples. Why do you think this is? Can you think of any strategies or approaches that might help to address this issue?

### 3.3 Multispectral Images

Apply your best model on multispectral images. You may use whichever image channels you wish, so long as you use more than just RGB (although you are not required to use any color channels). Calculate classification accuracy on the test data. Compare against results using RGB images.

How did adding multispectral channels impact your model's performance? Explain the role of additional spectral information in enhancing land cover classification.

## 4. Reflection Questions

What are your takeaways from tuning the parameters of the different models? What are your observations about increasing the number of training epochs? Did you run into any challenges or limitations when doing this? What was the impact of using dropout? How did the ensemble models compare to the other models? What kinds of challenges or limitations did you encounter when preparing and training the models for this assignment, and how might you address them in the future? How might you apply what you've learned about model tuning, dropout, and data processing to a different deep learning problem?